# Marlin-2B on Apple Silicon

[Marlin-2B](https://huggingface.co/NemoStation/Marlin-2B) is a 2B video VLM (a Qwen3.5-2B fine-tune) with two modes:

- **caption**: a scene paragraph plus timestamped events (`<start - end> description`).
- **find**: given a natural-language event, returns the span `(start, end)` where it happens.

This notebook is a feasibility spike for a camera alerting pipeline. It answers four questions:

1. Does Marlin run on a Mac (MPS), and how fast?
2. How much does cropping to the region of interest help, given Marlin's ~448×448 frame budget?
3. Can `find` be used to *detect* whether an event happened, or only to *locate* it?
4. What should the pipeline do with these results?

## Setup

Marlin is gated on Hugging Face: request access on the model page (auto-approved) and log in with `uv run hf auth login` before running this notebook.

In [ ]:
import math
import sys
import time

import cv2
import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForCausalLM
from transformers.video_utils import VideoMetadata

from supervision.assets import VideoAssets, download_assets

print("torch", torch.__version__, "| MPS available:", torch.backends.mps.is_available())

## Load the model

The model card loads with `device_map={"": "cuda"}`. On a Mac we load normally and move the model to `mps`; `device_map` would also need the `accelerate` package.

Qwen3.5 uses two kernels (`causal_conv1d`, `flash-linear-attention`) that are CUDA-only. On MPS, transformers falls back to reference PyTorch implementations, which work but are slower. That's a hard ceiling on Mac speed.

In [ ]:
t = time.time()
model = AutoModelForCausalLM.from_pretrained("NemoStation/Marlin-2B", trust_remote_code=True, dtype=torch.bfloat16)
model = model.to("mps").eval()
print(f"Loaded in {time.time() - t:.1f}s on {model.device} ({model.dtype})")

# The prompts and output parsers live in Marlin's remote code; reuse them so results match .caption() / .find().
marlin = sys.modules[type(model).__module__]
processor = model.processor

## Preparing clips without FFmpeg

Marlin's `.caption()` / `.find()` helpers only accept a file path and decode it with `torchcodec`, which needs the FFmpeg shared libraries installed system-wide. Instead we decode with OpenCV and hand the processor frames directly. That's also what a live pipeline would do: event clips come from frames already in memory.

We replicate Marlin's training-time preprocessing ourselves:

- **Sample at 2 fps** (the `FPS` default), capped at 240 frames (~2 minutes).
- **Resize each frame to at most 200,704 pixels** (`VIDEO_MAX_PIXELS`, ~448×448), with sides that are multiples of 32. The transformers video processor doesn't apply this per-frame cap by default; without it a 1080p clip costs ~4× more tokens and runs ~3× slower.
- **Optional crop** before resizing, so the pixel budget goes to the region we care about.

In [ ]:
SAMPLE_FPS = 2.0
MAX_FRAMES = 240
MAX_PIXELS = 200_704
FACTOR = 32  # patch size 16 × spatial merge 2


def load_clip(path, crop=None):
    capture = cv2.VideoCapture(str(path))
    source_fps = capture.get(cv2.CAP_PROP_FPS)
    total = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    keep = set(np.round(np.arange(0, total, source_fps / SAMPLE_FPS)).astype(int)[:MAX_FRAMES].tolist())

    frames = []
    for index in range(total):
        ok, frame = capture.read()
        if not ok:
            break
        if index in keep:
            if crop is not None:
                x1, y1, x2, y2 = crop
                frame = frame[y1:y2, x1:x2]
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

    height, width = frames[0].shape[:2]
    scale = min(1.0, math.sqrt(MAX_PIXELS / (height * width)))
    new_h = max(FACTOR, int(height * scale) // FACTOR * FACTOR)
    new_w = max(FACTOR, int(width * scale) // FACTOR * FACTOR)
    frames = [cv2.resize(f, (new_w, new_h), interpolation=cv2.INTER_AREA) for f in frames]
    if len(frames) % 2:  # frames are grouped in pairs (temporal patch size 2)
        frames.append(frames[-1])
    return np.stack(frames)


def generate(video, prompt, max_new_tokens):
    text = processor.apply_chat_template(
        [{"role": "user", "content": [{"type": "video"}, {"type": "text", "text": prompt}]}],
        tokenize=False,
        add_generation_prompt=True,
    )
    metadata = VideoMetadata(
        total_num_frames=len(video),
        fps=SAMPLE_FPS,
        width=video.shape[2],
        height=video.shape[1],
        duration=len(video) / SAMPLE_FPS,
        frames_indices=list(range(len(video))),
    )
    inputs = processor(
        text=[text],
        videos=[video],
        video_metadata=[metadata],
        return_tensors="pt",
        videos_kwargs={"do_sample_frames": False, "do_resize": False},
    ).to(model.device)

    prompt_tokens = inputs["input_ids"].shape[1]
    torch.mps.synchronize()
    start = time.time()
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    torch.mps.synchronize()
    elapsed = time.time() - start

    raw = processor.batch_decode(output[:, prompt_tokens:], skip_special_tokens=True)[0]
    return raw, {"input_tokens": prompt_tokens, "output_tokens": output.shape[1] - prompt_tokens, "seconds": elapsed}


def caption(video):
    raw, stats = generate(video, marlin.CAPTION_PROMPT, max_new_tokens=1024)
    _, scene, events = marlin.parse_caption(raw)
    return scene, events, stats


def find(video, event):
    raw, stats = generate(video, marlin.GROUNDING_PROMPT_TEMPLATE.format(event=event), max_new_tokens=64)
    return marlin.parse_span(raw)[1], stats

## Full frame vs. crop

The market square clip is portrait 4K (2160×3840). At Marlin's pixel budget the full frame becomes 320×576, so people are a few pixels tall. Cropping to the fountain area first (the same region as the zone in the supervision notebooks) gives the model ~6× more pixels per person.

In [ ]:
MARKET_SQUARE = download_assets(VideoAssets.MARKET_SQUARE, directory="data")
FOUNTAIN_CROP = (380, 860, 1820, 1920)  # x1, y1, x2, y2

timings = []
for name, crop in [("full frame", None), ("fountain crop", FOUNTAIN_CROP)]:
    video = load_clip(MARKET_SQUARE, crop)
    scene, events, stats = caption(video)
    timings.append({"call": f"caption, {name}", "frames": len(video), "frame size": f"{video.shape[2]}×{video.shape[1]}", **stats})

    print(f"=== {name} ({len(video)} frames at {video.shape[2]}×{video.shape[1]}) ===")
    print(scene)
    for event in events:
        print(f"  <{event['start']:.1f} - {event['end']:.1f}> {event['description']}")
    print()

The full-frame caption stays generic ("pedestrians walk across the square"). The crop describes individuals: clothing, where they walk, who stands still. For alerting, where jev has to judge *what a person did*, cropping around the tracked object or zone is essential.

Note the scene paragraph can contain stock phrases that aren't true (e.g. "the camera slowly pulls back" on a fixed camera). Rules that consume captions should rely on the events, not on camera descriptions.

## Can `find` detect events?

`find` is trained to return a span for an event it's told is in the video. What happens when the event isn't there? We ask for two events that do happen in the fountain crop and four that don't.

In [ ]:
video = load_clip(MARKET_SQUARE, FOUNTAIN_CROP)

QUERIES = [
    ("a person in a white jacket walks across the square", True),
    ("a person stands next to the fountain", True),
    ("a person falls to the ground", False),
    ("a car drives through the square", False),
    ("a dog runs across the square", False),
    ("a person climbs into the fountain", False),
]

rows = []
for query, happens in QUERIES:
    span, stats = find(video, query)
    rows.append({"query": query, "actually happens": happens, "find span (s)": span, "seconds": round(stats["seconds"], 1)})
timings.append({"call": "find, fountain crop", "frames": len(video), "frame size": f"{video.shape[2]}×{video.shape[1]}", **stats})

pd.DataFrame(rows)

Every query gets a span, including events that never happen. We also checked (outside this notebook) two ways to recover a "not found" signal:

- **Token confidence** of the `find` answer: average log-probabilities are indistinguishable between true and false queries (−0.2 to −0.4 for both).
- **A yes/no prompt** ("Does this happen in the video? Answer yes or no."): Marlin ignores it and answers in its trained caption or find format. The fine-tune has replaced general instruction following.

So `find` answers *when*, never *whether*.

## Timings

In [ ]:
pd.DataFrame(timings).round(1)

Model load takes ~5 s and uses about 6 GB of unified memory. With correct preprocessing, a caption of an 8-second clip takes under 10 s and a find about 3.5 s. Cost scales with the number of frames (2 per second of video) and with output length, so a 2-minute clip will take several times longer.

## Takeaways for the alerting pipeline

1. **Marlin runs on a Mac.** Loading needs `.to("mps")` instead of `device_map`, and frames should be passed directly (no FFmpeg dependency). Expect roughly real-time or slower for captions: fine for a handful of events per minute on one camera, not for continuous captioning.
2. **Crop before captioning.** Use the tracked object's box (with margin) or the zone's bounding box. Full-frame captions of wide cameras are too coarse for per-person judgements.
3. **Do our own preprocessing.** 2 fps sampling and the ~200k pixel cap per frame must be applied explicitly; the defaults cost ~3× more time.
4. **Don't alert on `find` alone.** It always returns a span. Use it to locate a moment after something else has decided the event happened, or verify its span: caption the segment it returns and let jev decide.
5. **Keep clips under the 2-minute cap.** Dwell or "until leave" clips longer than that need to be chunked or subsampled.